# 07 — Artifact 2: Statin Cross-Class Divergence

🔧 Script · 📚 Reference

**Artifact 2:** Rhett759 is on simvastatin (Synthea) and atorvastatin (Epic). These are *different* drugs in the same therapeutic class (HMG-CoA reductase inhibitors — statins). The harmonizer does NOT merge them; it surfaces a `CrossClassFlag` and attaches `EXT_CONFLICT_PAIR` to both resources.


## 1. Build the two silver-tier MedicationRequests

In [ ]:
from ehi_atlas.harmonize.provenance import SYS_SOURCE_TAG, SYS_LIFECYCLE

# Synthea uses RxCUI 316672 (product-level SCD: "Simvastatin 10 MG Oral Tablet")
med_synthea = {
    "resourceType": "MedicationRequest",
    "id": "synthea-med-simvastatin",
    "meta": {"tag": [
        {"system": SYS_SOURCE_TAG, "code": "synthea"},
        {"system": SYS_LIFECYCLE,  "code": "standardized"},
    ]},
    "status": "active",
    "intent": "order",
    "medicationCodeableConcept": {
        "coding": [{"system": "http://www.nlm.nih.gov/research/umls/rxnorm",
                    "code": "316672", "display": "Simvastatin 10 MG Oral Tablet"}],
        "text": "Simvastatin 10 MG Oral Tablet",
    },
    "subject": {"reference": "Patient/rhett759"},
    "authoredOn": "2022-06-01",
}

# Epic uses RxCUI 83367 (ingredient-level: atorvastatin), discontinued
med_epic = {
    "resourceType": "MedicationRequest",
    "id": "epic-med-atorvastatin",
    "meta": {"tag": [
        {"system": SYS_SOURCE_TAG, "code": "epic-ehi"},
        {"system": SYS_LIFECYCLE,  "code": "stub-silver"},
    ]},
    "status": "stopped",
    "intent": "order",
    "medicationCodeableConcept": {
        "coding": [{"system": "http://www.nlm.nih.gov/research/umls/rxnorm",
                    "code": "83367", "display": "atorvastatin"}],
        "text": "atorvastatin",
    },
    "subject": {"reference": "Patient/rhett759"},
    "authoredOn": "2024-01-15",
    "dispenseRequest": {"validityPeriod": {"start": "2024-01-15", "end": "2025-09-01"}},
}

print("Built simvastatin (RxCUI 316672, active) and atorvastatin (RxCUI 83367, stopped)")


## 2. episode_from_medication_request

In [ ]:
# 🔧 Script
from ehi_atlas.harmonize.medication import episode_from_medication_request

ep_simva = episode_from_medication_request(med_synthea)
ep_atorva = episode_from_medication_request(med_epic)

print("Simvastatin episode:")
print(f"  rxcui={ep_simva.rxcui}  status={ep_simva.status}  label={ep_simva.ingredient_label}")

print()
print("Atorvastatin episode:")
print(f"  rxcui={ep_atorva.rxcui}  status={ep_atorva.status}  period={ep_atorva.period_start} → {ep_atorva.period_end}")


## 3. episodes_same_ingredient → False

In [ ]:
from ehi_atlas.harmonize.medication import episodes_same_ingredient

same = episodes_same_ingredient(ep_simva, ep_atorva)
print("episodes_same_ingredient(simvastatin, atorvastatin):", same)
print("→ Different ingredients → kept as separate gold episodes")


## 4. detect_cross_class_flags

In [ ]:
from ehi_atlas.harmonize.medication import detect_cross_class_flags

flags = detect_cross_class_flags([ep_simva, ep_atorva])
print(f"Cross-class flags found: {len(flags)}")
if flags:
    f = flags[0]
    # ingredient_a / ingredient_b are RxCUI strings; sources_a / sources_b are source-tag lists
    print(f"  ingredient_a (RxCUI): {f.ingredient_a}  sources: {f.sources_a}")
    print(f"  ingredient_b (RxCUI): {f.ingredient_b}  sources: {f.sources_b}")
    print(f"  class_label: {f.common_class_label}")


## 5. detect_medication_class_conflicts + apply_conflict_pairs

In [ ]:
from ehi_atlas.harmonize.conflict import detect_medication_class_conflicts, apply_conflict_pairs
from dataclasses import dataclass

# Adapt CrossClassFlag to the protocol shape expected by conflict.py
@dataclass
class AdaptedFlag:
    ingredient_a: str
    ingredient_b: str
    class_label: str
    source_a: str
    source_b: str
    resource_a_reference: str
    resource_b_reference: str

if flags:
    f = flags[0]
    adapted = [AdaptedFlag(
        ingredient_a=f.ingredient_a,
        ingredient_b=f.ingredient_b,
        class_label=f.common_class_label,
        source_a=ep_simva.source_tag or "synthea",
        source_b=ep_atorva.source_tag or "epic-ehi",
        resource_a_reference=f"MedicationRequest/{med_synthea['id']}",
        resource_b_reference=f"MedicationRequest/{med_epic['id']}",
    )]

    conflicts = detect_medication_class_conflicts(adapted)
    print(f"ConflictPairs detected: {len(conflicts)}")
    if conflicts:
        cp = conflicts[0]
        print(f"  kind:    {cp.kind}")
        print(f"  label:   {cp.label}")
        print(f"  summary: {cp.summary}")

    # Apply symmetric conflict-pair extensions to both resources
    resources_by_ref = {
        f"MedicationRequest/{med_synthea['id']}": med_synthea,
        f"MedicationRequest/{med_epic['id']}": med_epic,
    }
    apply_conflict_pairs(conflicts, resources_by_ref)

    from ehi_atlas.harmonize.provenance import EXT_CONFLICT_PAIR
    for med, name in [(med_synthea, "simvastatin"), (med_epic, "atorvastatin")]:
        ext = next((e for e in med.get("extension", []) if e.get("url") == EXT_CONFLICT_PAIR), None)
        print(f"  {name} EXT_CONFLICT_PAIR → {ext.get('valueReference', {}).get('reference') if ext else 'not set'}")


## 6. Note: two-part bug fix (task 3.11)

The integration test found two issues and fixed them inline:
1. Synthea emits RxCUI **316672** (product-level SCD) not 36567 (ingredient-level). Added 316672 to `_RXCUI_CLASS_LABEL`.
2. `apply_conflict_pairs` mutated the silver dict, but the merged gold dict is a new object. The orchestrator now propagates `EXT_CONFLICT_PAIR` from silver to gold after the merge step.

Both fixes are in `ehi_atlas/harmonize/medication.py` and `ehi_atlas/harmonize/orchestrator.py`.


## 7. Confirm in the actual gold bundle

In [ ]:
from pathlib import Path
import json

ATLAS_ROOT = Path("..").resolve()
gold_bundle = json.loads(
    (ATLAS_ROOT / "corpus" / "gold" / "patients" / "rhett759" / "bundle.json").read_text()
)

from ehi_atlas.harmonize.provenance import EXT_CONFLICT_PAIR

statin_meds = []
for entry in gold_bundle["entry"]:
    res = entry["resource"]
    if res.get("resourceType") != "MedicationRequest":
        continue
    codings = res.get("medicationCodeableConcept", {}).get("coding", [])
    rxcuis = {c.get("code") for c in codings}
    if rxcuis & {"316672", "83367", "36567"}:
        statin_meds.append(res)

print(f"Statin MedicationRequests in gold: {len(statin_meds)}")
for med in statin_meds:
    codings = med.get("medicationCodeableConcept", {}).get("coding", [])
    rxcui = next((c.get("code") for c in codings), "?")
    has_conflict = any(e.get("url") == EXT_CONFLICT_PAIR for e in med.get("extension", []))
    print(f"  {med['id']}  RxCUI={rxcui}  status={med['status']}  conflict_pair={has_conflict}")


**Next:** [08_layer3_observation_artifact_5.ipynb](./08_layer3_observation_artifact_5.ipynb)